# Track A / Track B diagnostic: does SMI itself scale with trial count?

**Context**: Phase 4's Track A paired within-cell comparison (JSY093, DCZ1/DCZ2/DCZ3) found DCZ SMI *higher* than saline SMI in the tracked, jointly-reliable cells -- the opposite direction from Track B's population-level finding (DCZ lower, significant in 4/5 groups). All three Track A groups agree in direction (DCZ1: +0.074, DCZ2: +0.150, DCZ3: +0.114 mean per-cell change), with DCZ3 significant on both Wilcoxon (p=0.035) and paired t-test (p=0.025).

**Question this notebook tests**: DCZ sessions have systematically more trials than their paired saline sessions. Function 4.5/4.6 (`4.SessionComparison.py`) already tested whether trial count explains the saline-vs-DCZ gap in *reliability fraction* -- but never tested whether SMI's own *magnitude* trends with trial count (e.g. more trials -> cleaner tuning-curve estimate -> inflated SMI, independent of any real condition effect). If it does, that's a more parsimonious explanation for Track A's reversed direction than a genuine biological effect.

**Design**: same WLS calibration-check architecture as Function 4.5/4.6 (`smi_summary ~ n_trials + C(session_type)`), just with SMI (not reliability fraction) as the outcome, run across the *whole* session catalog (baseline/saline/dcz, trial-count range ~10-118) rather than just the 3 tracked-cell groups -- gives real trial-count variation to regress against instead of 2 discrete values per group.

Reimplements small loading utilities locally rather than importing from `4.SessionComparison.py` (digit-prefixed module names aren't importable -- same convention used throughout this pipeline).

In [ ]:
import os
import re
import glob
from collections import Counter

import numpy as np
import h5py
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import statsmodels.formula.api as smf

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

## Function A -- discover sessions with saved SMI results

Reimplemented from `4.SessionComparison.py`'s Function 4.2.

In [ ]:
def discover_smi_sessions_local(animal_dir):
    """
    Scan animal_dir for every already-computed *_smi_results_dreadd.h5
    file, labeling each by whichever known naming pattern its TSeries
    folder matches. Reimplemented from 4.SessionComparison.py's Function
    4.2 (digit-prefixed module names aren't importable, same convention
    used throughout this pipeline).

    Parameters
    ----------
    animal_dir : str

    Returns
    -------
    catalog : dict
        {label: {'save_path': str, 'session_type': str, 'tseries_dir': str}}
    """
    save_paths = sorted(glob.glob(os.path.join(animal_dir, '**', '*_smi_results_dreadd.h5'),
                                   recursive=True))

    entries = []
    unmatched = []

    for save_path in save_paths:
        tseries_dir = os.path.dirname(save_path)
        tseries_name = os.path.basename(tseries_dir)
        parent_dir = os.path.dirname(tseries_dir)
        parent_name = os.path.basename(parent_dir)

        upper_tseries = tseries_name.upper()

        if 'SAL' in upper_tseries:
            session_type = 'saline'
            base_label = f'{parent_name}_SALINE'
        elif 'DCZ' in upper_tseries:
            session_type = 'dcz'
            base_label = f'{parent_name}_DCZ'
        else:
            day_match = re.search(r'Day(\d+)', parent_name, re.IGNORECASE)
            if day_match:
                session_type = 'baseline'
                base_label = f'Day{day_match.group(1)}'
            else:
                session_type = 'unknown'
                base_label = tseries_name
                unmatched.append(base_label)

        entries.append((base_label, session_type, save_path, tseries_dir, tseries_name))

    label_counts = Counter(e[0] for e in entries)

    catalog = {}
    for base_label, session_type, save_path, tseries_dir, tseries_name in entries:
        label = f'{base_label}__{tseries_name}' if label_counts[base_label] > 1 else base_label
        if label in catalog:
            print(f"WARNING: label '{label}' still collides after disambiguation -- "
                  f"keeping {catalog[label]['save_path']}, skipping {save_path}")
            continue
        catalog[label] = {'save_path': save_path, 'session_type': session_type, 'tseries_dir': tseries_dir}

    print(f"Discovered {len(catalog)} sessions with saved SMI results under {animal_dir}:")
    for label, info in catalog.items():
        print(f"  [{info['session_type']:>8}] {label}")

    if unmatched:
        print(f"\n{len(unmatched)} session(s) didn't match a known naming pattern: {unmatched}")

    return catalog

## Function B -- load one session's saved SMI results

Reimplemented from Function 4.1 (`load_session_smi_for_comparison`).

In [ ]:
def load_session_smi_local(save_path, session_label=None):
    """
    Load one session's already-computed *_smi_results_dreadd.h5 into a
    tidy per-cell DataFrame. Reimplemented from 4.SessionComparison.py's
    Function 4.1.

    Parameters
    ----------
    save_path : str
    session_label : str, optional

    Returns
    -------
    df : pandas.DataFrame
        One row per cell: cell_idx, SMI, valid, analysis_reliable,
        combined_reliable, avg_cc, cohen_d, layer.
    """
    with h5py.File(save_path, 'r') as f:
        stored_label = f.attrs.get('session_label', os.path.basename(save_path))
        SMI_values = f['global_smi/SMI_all_cells'][:]
        valid_cells_mask = f['global_smi/valid_cells_mask'][:]
        analysis_reliable_cells = f['global_smi/analysis_reliable_cells'][:]
        combined_reliable = f['reliability/combined_reliable'][:]
        avg_cc = f['reliability/avg_cc'][:]
        cohen_d = f['reliability/cohen_d'][:]

        n_cells = len(SMI_values)
        layer_of_cell = np.full(n_cells, None, dtype=object)
        for safe_name in f['layer_smi']:
            layer_grp = f['layer_smi'][safe_name]
            original_name = layer_grp.attrs.get('original_name', safe_name)
            cell_indices = layer_grp['cell_indices'][:]
            layer_of_cell[cell_indices] = original_name

    df = pd.DataFrame({
        'cell_idx': np.arange(n_cells),
        'SMI': SMI_values,
        'valid': valid_cells_mask,
        'analysis_reliable': analysis_reliable_cells,
        'combined_reliable': combined_reliable,
        'avg_cc': avg_cc,
        'cohen_d': cohen_d,
        'layer': layer_of_cell,
    })
    df['session_label'] = session_label if session_label is not None else stored_label

    return df

## Function C -- read a session's trial count

Reimplemented from Function 4.5 (`get_session_trial_count`).

In [ ]:
def get_session_trial_count_local(tseries_dir):
    """
    Read a session's trial count straight from its preproc.h5.
    Reimplemented from 4.SessionComparison.py's Function 4.5.

    Parameters
    ----------
    tseries_dir : str

    Returns
    -------
    n_trials : int
    """
    preproc_files = glob.glob(os.path.join(tseries_dir, "*preproc*.h5"))
    if not preproc_files:
        raise FileNotFoundError(f"No *preproc*.h5 found in {tseries_dir}")
    with h5py.File(preproc_files[0], 'r') as f:
        n_trials = f['spatial_activity'].shape[1]
    return n_trials

## Function D -- build the SMI-vs-trial-count calibration table

Same shape as Function 4.5's `build_reliability_trial_count_calibration`, y-axis swapped from reliable_fraction to SMI.

In [ ]:
def build_smi_trial_count_calibration(session_catalog, summary_stat='median'):
    """
    For every session with saved SMI results: n_trials, session_type, and
    a summary (median or mean) SMI among 'valid' cells (reliable_valid_
    cells -- same filter Function 4.7 uses for the actual hypothesis
    test), plus n_valid_cells (for weighting the regression below). Same
    shape as 4.SessionComparison.py's Function 4.5
    (build_reliability_trial_count_calibration), y-axis swapped from
    reliable_fraction to SMI.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions_local.
    summary_stat : str
        'median' or 'mean'.

    Returns
    -------
    calib_df : pandas.DataFrame
        One row per session: session_label, session_type, n_trials,
        n_valid_cells, smi_summary, predicted_smi, residual.
    coeffs : numpy.ndarray
        [slope, intercept] of the fitted line (smi_summary ~ n_trials).
    """
    stat_fn = np.median if summary_stat == 'median' else np.mean

    rows = []
    skipped = []
    for label, info in session_catalog.items():
        try:
            n_trials = get_session_trial_count_local(info['tseries_dir'])
        except FileNotFoundError:
            skipped.append(label)
            continue

        session_df = load_session_smi_local(info['save_path'], session_label=label)
        valid_smi = session_df.loc[session_df['valid'], 'SMI'].to_numpy()

        rows.append({
            'session_label': label,
            'session_type': info['session_type'],
            'n_trials': n_trials,
            'n_valid_cells': len(valid_smi),
            'smi_summary': float(stat_fn(valid_smi)) if len(valid_smi) > 0 else np.nan,
        })

    if skipped:
        print(f"WARNING: {len(skipped)} session(s) skipped -- no *preproc*.h5 found: {skipped}")

    calib_df = pd.DataFrame(rows).dropna(subset=['smi_summary']).reset_index(drop=True)

    coeffs = np.polyfit(calib_df['n_trials'], calib_df['smi_summary'], deg=1)
    calib_df['predicted_smi'] = np.polyval(coeffs, calib_df['n_trials'])
    calib_df['residual'] = calib_df['smi_summary'] - calib_df['predicted_smi']

    print(f"Fitted trend: {summary_stat}_SMI ~ {coeffs[0]:.6f} * n_trials + {coeffs[1]:.4f}")
    print(calib_df.sort_values('n_trials')[
        ['session_label', 'session_type', 'n_trials', 'n_valid_cells', 'smi_summary', 'residual']
    ].to_string(index=False))

    return calib_df, coeffs

## Function E -- test whether condition predicts SMI beyond trial count

Same design as Function 4.6's `test_condition_effect_on_reliability`, applied to SMI directly.

In [ ]:
def test_condition_effect_on_smi(calib_df, weight_col='n_valid_cells', reference_level='baseline'):
    """
    Weighted least squares: smi_summary ~ n_trials + C(session_type),
    weighted by weight_col. Same design as 4.SessionComparison.py's
    Function 4.6 (test_condition_effect_on_reliability), applied to SMI
    directly instead of a reliability-component fraction.

    Parameters
    ----------
    calib_df : pandas.DataFrame
        From build_smi_trial_count_calibration.
    weight_col : str
    reference_level : str

    Returns
    -------
    model_result : statsmodels regression results object
    """
    df = calib_df.copy()
    other_levels = [c for c in df['session_type'].unique() if c != reference_level]
    df['session_type'] = pd.Categorical(df['session_type'], categories=[reference_level] + other_levels)

    formula = "smi_summary ~ n_trials + C(session_type)"
    model_result = smf.wls(formula, data=df, weights=df[weight_col]).fit()

    print(f"\n=== smi_summary ~ n_trials + condition (weighted by {weight_col}, "
          f"reference='{reference_level}') ===")
    print(model_result.summary().tables[1])

    param_names = list(model_result.params.index)
    dcz_name = next((p for p in param_names if 'dcz' in p.lower()), None)
    saline_name = next((p for p in param_names if 'saline' in p.lower()), None)
    if dcz_name and saline_name:
        contrast = f"{dcz_name} - {saline_name}"
        print(f"\nDirect DCZ vs. saline contrast:")
        print(model_result.t_test(contrast))

    return model_result

## Function F -- plot the calibration

Same design as Function 4.5's `plot_reliability_calibration`.

In [ ]:
def plot_smi_trial_count_calibration(calib_df, coeffs):
    """
    Scatter of n_trials vs. smi_summary, colored by session_type, with the
    fitted trend line overlaid. Same design as Function 4.5's
    plot_reliability_calibration.

    Parameters
    ----------
    calib_df : pandas.DataFrame
    coeffs : numpy.ndarray

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    color_by_type = {'baseline': 'tab:blue', 'saline': 'tab:orange', 'dcz': 'tab:green'}

    fig, ax = plt.subplots(figsize=(9, 8))
    for session_type, group in calib_df.groupby('session_type'):
        ax.scatter(group['n_trials'], group['smi_summary'],
                   c=color_by_type.get(session_type, 'gray'), s=80, alpha=0.8,
                   label=session_type)

    x_line = np.linspace(calib_df['n_trials'].min(), calib_df['n_trials'].max(), 100)
    ax.plot(x_line, np.polyval(coeffs, x_line), color='black', linestyle='--', label='linear fit')

    ax.set_xlabel('Trial count')
    ax.set_ylabel('Median SMI (valid cells)')
    ax.set_title('SMI vs. trial count')
    ax.legend(loc='best')

    plt.tight_layout()
    return fig

## Run it -- JSY093

In [ ]:
ANIMAL_DIR = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD"

session_catalog = discover_smi_sessions_local(ANIMAL_DIR)
calib_df, coeffs = build_smi_trial_count_calibration(session_catalog)
model_result = test_condition_effect_on_smi(calib_df)
fig = plot_smi_trial_count_calibration(calib_df, coeffs)
plt.show()